In [ ]:
# ── IdiomBERT Exp 03 (MuRIL Joint canonical) ─────────────────────────────────
# FIRST: Runtime → Change runtime type → T4 GPU.
#
# Trains System E architecture with google/muril-base-cased encoder (SentencePiece).
# Single seed 42. Flip-2 predictions exist but are non-canonical (partial run).
# This is the full EN+ES+HI+TE training for the canonical rigor_muril_joint_s42 key.
#
# After training: run Cell 1 to register results.
# Expected: MuRIL helps HI/TE; ES may drop (acceptable — MuRIL not Iberian-focused).
import os, subprocess
from pathlib import Path

# 1. GPU check
assert subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0, \
    'No GPU — Runtime → Change runtime type → T4 GPU'

# 2. Clone or pull repo
REPO = '/content/Idiomator_Research'
if Path(REPO).exists():
    !cd $REPO && git pull --ff-only
else:
    !git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git $REPO
%cd $REPO
!pip install -q -r Requirements.txt

# 3. Mount Drive
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
os.environ['DRIVE_OUT'] = DRIVE_OUT
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)

# 4. Durability probe
probe = Path(DRIVE_OUT) / '.write_test'
probe.write_text('ok')
assert probe.read_text() == 'ok', 'Drive not writable'
probe.unlink()
print('✓ Drive writable')

# 5. Train
LOG = f'{DRIVE_OUT}/exp03_muril_console.log'
!bash experiments/rigor/run_03_muril_joint.sh 2>&1 | tee -a "$LOG"
print(f'\nDone. Log at {LOG}')

In [ ]:
# ── Register canonical MuRIL results in pipeline_eval_results.json ───────────
# Overwrites the non-canonical rigor_muril_joint_s42 key (from flip-2 run)
# with the full canonical training results.
import os
from pathlib import Path

DRIVE_OUT = os.environ.get('DRIVE_OUT', '/content/drive/MyDrive/IdiomatorRigor')
MODEL_DIR = 'models/rigor_joint_muril_full'
PREDS     = f'{MODEL_DIR}/test_predictions.jsonl'

# Verify Drive-backed symlink survived
assert os.path.islink(MODEL_DIR), f'{MODEL_DIR} not a symlink — training output may be ephemeral'
assert os.path.exists(MODEL_DIR), f'{MODEL_DIR} symlink target missing on Drive'
assert Path(PREDS).exists(),      f'{PREDS} missing — did training complete?'
print(f'✓ {MODEL_DIR} -> {os.path.realpath(MODEL_DIR)}')
print(f'✓ predictions at {PREDS}')

# Re-read from Drive to confirm durability
n_preds = sum(1 for _ in open(PREDS))
print(f'✓ {n_preds} prediction rows on Drive')

# Register — overwrites rigor_muril_joint_s42 with canonical results
!python Evaluation/Full_evaluation.py \
    --muril_joint_preds "$PREDS" \
    2>&1 | tail -50

# Confirm key landed
import json
result = json.loads(Path('results/pipeline_eval/pipeline_eval_results.json').read_text())
assert 'rigor_muril_joint_s42' in result, 'Key not found in results json'
m = result['rigor_muril_joint_s42']
print('\n✓ rigor_muril_joint_s42 registered:')
for k, v in m.items():
    if isinstance(v, dict) and 'joint_f1' in v:
        print(f'  {k}: joint_f1={v["joint_f1"]}')

In [ ]:
# ── MuRIL vs mBERT System E comparison ───────────────────────────────────────
# Quick sanity check: MuRIL should help HI/TE, may drop ES.
import json
from pathlib import Path

result = json.loads(Path('results/pipeline_eval/pipeline_eval_results.json').read_text())

SYSTEMS = [
    ('system_e_mbert_pipeline', 'E (mBERT)'),
    ('rigor_muril_joint_s42',   'E (MuRIL)'),
]

LANGS = ['English', 'Spanish', 'Hindi', 'Telugu', 'Indonesian']

print(f'{'':20}', end='')
for lang in LANGS:
    print(f'{lang:>11}', end='')
print(f'{'Overall':>11}')

for key, label in SYSTEMS:
    if key not in result:
        print(f'{label:20}  (not in results)')
        continue
    sys_r = result[key]
    print(f'{label:20}', end='')
    vals = []
    for lang in LANGS:
        jf = sys_r.get(lang, {}).get('joint_f1')
        s = f'{jf:.4f}' if jf is not None else '  n/a'
        print(f'{s:>11}', end='')
        if jf is not None and lang in ['English','Spanish','Hindi','Telugu']:
            vals.append(jf)
    macro = sum(vals)/len(vals) if vals else None
    print(f'  {macro:.4f}' if macro else '')

print('\nExpected: MuRIL > mBERT on HI/TE; ES may drop; EN comparable.')